# Calibration technique comparison — independent optimisers vs EFC

<details>
<summary>The independent-optimiser baseline for reviewer comment R1-C1: emcee and multi-start
gradient descent against EFC, on one identical calibration task</summary>

Reviewer 1 asked for "at least one comparison against an established optimization method on an
identical calibration task". This notebook builds it from the two methods the reviewer named:

| lane | what it is |
|---|---|
| **emcee** | affine-invariant ensemble MCMC, controllers off, 128 walkers |
| **gd** | multi-start gradient descent (Adam), controllers off, 64 restarts |
| **EFC** | the embedded feedback controller — one staged calibration *is* one simulation |

All three solve the same problem: the same 16 model parameters, driven to the same 16 physiological
twin targets, over the same prior box, under the same explicit-Euler integration. The MCMC and GD
fits were produced before `sepsis.json`/`cvModel.json` were renamed to `sepsis_cubic.json`/
`cvModel_cubic.json` (commit `d2f3d0e`, a pure rename — contents byte-identical), so their stored
provenance names the old files; the setup cell prints this so it is visible rather than assumed,
and asserts the target vectors match.

This notebook runs **no forward solve**: every quantity is read from stored `.h5` data, so it
re-plots instantly after a kernel restart. Set `runConfig["paper"]["emit"]` to write the LaTeX
tables and the figure into the paper tree.

Every knob lives in the `runConfig` dict below (project single-config-surface rule).

</details>

In [ ]:
# region -> runConfig (the ONE place run configuration lives; project single-config-surface rule)
# runConfig — the ONE place run configuration lives (project rule: see repo-root CLAUDE.md).
runConfig = {
    # --- the two independent baselines the reviewer named (R1-C1) ---
    "methods": [
        # `independent`: are this method's attempts independent calibrations? GD restarts and EFC
        # starts are -- each is a self-contained answer, so the spread across them is a robustness
        # statistic. An ensemble sampler's walkers are NOT: the proposal for one walker is built
        # from the positions of the others, so 128 walkers are one chain, and their spread is
        # posterior width, not a success rate. The per-attempt columns are suppressed where False.
        {"name": "emcee", "path": "notebookData/convergence/mcmc_baseline2_sepsis_emcee.h5",
         "label": "MCMC", "attempts": "walkers", "color": "C0", "independent": False},
        {"name": "gd",    "path": "notebookData/convergence/mcmc_baseline_sepsis_gd.h5",
         "label": "GD", "attempts": "restarts", "color": "C1", "independent": True},
    ],

    # --- EFC lane (consult-only: one stored population file) ---
    # Only populations are compared: N independent random starts driven to the SAME fixed
    # targets, the like-for-like counterpart of GD's restarts and emcee's walkers.
    # The control law and the stage budget are read from the file's OWN stored config, not
    # from its filename -- several population files on disk are misnamed.
    "efc": {
        # One entry per control law. Each is an independent multi-start population driven to the
        # SAME targets as the baselines; the law and the stage budget are read from the file's own
        # stored config (several population files on disk are misnamed), so `name` is only the key.
        "populations": [
            # `label` overrides the display name only. The control LAW is still read from each
            # file's stored config and cross-checked against the label -- a mismatch prints a NOTE.
            {"name": "a", "path": "notebookData/convergence/population_linear_batch.h5", "color": "C2",
             "label": "EFC (linear)"},
            {"name": "b", "path": "notebookData/convergence/population_cubic_batch.h5",  "color": "C4",
             "label": "EFC (cubic)"},
        ],
        "cloudThresh": 2.0,   # population runs under this max|rel err| (%) form the identifiability cloud
        # EFC has no optimiser iterations -- one staged calibration IS one simulation -- so its
        # convergence trace is read from this stored trajectory dataset ("raw_coarse" = the
        # decimated trace, "raw" = every step; both carry all 16 targeted observables).
        "trajDataset": "raw_coarse",
    },

    # --- scenario / model naming the shared task (no forward solve here) ---
    "scenario": "sepsis_cubic.json",    # was sepsis.json   before commit d2f3d0e (identical content)
    "model":    "cvModel_cubic.json",   # was cvModel.json  before commit d2f3d0e (identical content)

    # --- analysis / plot ---
    "analysis": {
        "atm":             760.0,   # atmospheric offset for absolute-pressure signals (gauge = raw - atm)
        "solveWall":       0.134,   # measured wall s per 10 s Euler solve (solver_compare
                                    #   integrator table: dt 5e-4, one lane, one CPU)
        "countSettleRuns": False,   # False -> every method is costed on ONE 10 s settle run
                                    #   (the archived emcee file stores settleRuns 3, GD and
                                    #   the EFC populations 1); True uses each stored value
        "divergenceLimit": 1e6,     # |value| >= this in any EFC observable/param = out of scope
        "errBand":         1.0,     # +/- target-tolerance band (%) drawn on the error plots
        "convBand":        [25, 75],# convergence-plot shaded band: percentiles over the
                                    #   independent attempts (walkers / restarts / EFC lanes)
        "credibleMass":    0.68,    # caterpillar bar = this central posterior mass (0.68 = +/-1 sigma)
        "successLevels":   [0.5, 1.0, 2.0, 5.0],   # one "attempts within X%" table row each
        "successThresh":   2.0,     # threshold for the figure panels and the cloud, not the table
        "fontSize":        8,       # base plot font (pt) at paper.figSize; panel titles use it +1.
                                    #   The comparison figure is drawn AT the manuscript column
                                    #   width, so this is the final rendered size -- not a big
                                    #   figure downscaled (which renders 7 pt text at 2 pt).
        "convergedLimit":  1000.0,  # max|rel err| (%) above which an attempt DID NOT CONVERGE.
                                    #   Treated exactly like a NaN / sentinel-stamped lane: dropped
                                    #   from every panel and table, counted and printed. None keeps
                                    #   every attempt (the values then show as huge errors instead).
        "foldClip":        1000.0,  # panel (b) y-cap: fold differences above this are hidden from
                                    #   the panel and counted in-figure. Cosmetic only -- a capped
                                    #   attempt still appears in (a), (c) and every table.
        "errorScale":      "symlog",# panel (a) y-scale: "symlog" (linear inside +/- errBand,
                                    #   log outside -- keeps the sub-1% boxes readable next to a
                                    #   -12% outlier) or "linear".
        "ylim": {                   # per-panel axis limits; None = fit to the data
            "fitError": None,       #   (a) relative error (%)      e.g. [-15, 12]
            "fold":     2,       #   (b) fold-difference top     e.g. 30 (still capped by foldClip)
        },
    },

    # --- optional: emit the paper artifacts (flip to True for an emit run, then revert) ---
    "paper": {
        # Per-artifact emit flags: each block below writes only when ITS flag is True, so the
        # appendix table can be refreshed without also rewriting the Results-section artifacts
        # (which are pinned to a different reviewer comment and to a different lane set).
        "emit": {"table": True, "setupTable": True, "appendixSetup": True,
                 "figure": True, "figureConvergence": True},
        "dir":        "EFC_Paper/revision/generated",           # generated LaTeX
        "imageDir":   "EFC_Paper/revision/Images",              # generated figures
        "table":      "techniqueComparison.tex",                # non-float -> Results
        # LaTeX size command applied inside the cost table's brace group ("" disables it).
        # Two steps up from scriptsize: the sample-size row carries bare counts and the
        # method headers stack onto two lines, so the table reads at \small in the 8.5 cm
        # column. The widest row (Total solves) sets the ceiling.
        "tableFontSize": "small",
        "setupTable": "techniqueSetup.tex",                     # non-float, all lanes -> Results
        # Appendix variant: the two INDEPENDENT baselines only, emitted as a non-float so it is
        # legal inside the manuscript's two-column `multicols` body (a real float is silently
        # dropped there). EFC lanes are excluded -- the appendix describes the baselines.
        "appendixSetup":     "baselineSetup.tex",
        # must match the `setupCols` labels built in the setup-table cell (the two baselines)
        "appendixSetupLanes": ["MCMC", "GD"],
        # TWO figures -> Results. The comparison figure carries (a) fit accuracy and
        # (b) parameter agreement; the convergence trace is emitted separately because it is
        # read against the cost table, and because all three panels stacked at column width
        # made a figure taller than the text block.
        "figure":            "techniqueComparison.png",
        "figureConvergence": "methodConvergence2.png",
        # inches, included at \includegraphics[width=8.5cm]. Only the ASPECT RATIO matters -- the
        # whole figure is scaled to the column, so the on-page font size is fontSize x 3.35/width.
        # 32 labelled rows over a column-height figure put the ceiling near 8 pt: past that the
        # y tick labels collide, and buying more row pitch makes the figure taller than the page.
        "figSize":      [3.35, 7.8],
        "heightRatios": [1.0, 0.95],        # panel (a) : (b)
        "figSizeConvergence": [3.35, 2.6],  # the convergence figure, same column width

        # --- wide variant of the comparison figure (its own cell, its own file) ---
        # Same two panels and the same data, drawn at the manuscript's full text width instead of
        # the 8.5 cm column. A horizontal box's LENGTH lies along the data axis, so doubling the
        # width doubles the page a given spread covers: the tightest population (EFC linear) stops
        # rendering as a hairline. Emitted to its own file so both figure cells can be run in any
        # order without clobbering each other.
        "figureWide":    "techniqueComparisonWide.png",
        "figSizeWide":   [6.9, 6.6],   # inches, included at \includegraphics[width=\linewidth]
        "markScaleWide": 2.0,          # box/whisker/median line widths and point sizes x this
        "boxFracWide":   0.95,         # box thickness as a fraction of its slot (column cell: 0.85)
        "fontSizeWide":  8,            # the wide figure renders 1:1, so this is the on-page pt size
        # EFC (linear) has a per-target IQR of ~0.009% against ~0.65% for MCMC, so its box renders a
        # couple of pixels wide at any canvas size. Its median is therefore marked explicitly, and
        # the tight populations get proportionally thicker rows (slots are reallocated to keep the
        # series tiling one row). Any label absent from rowScaleWide keeps a scale of 1.0.
        "medianMarkerWide":     "D",
        "medianMarkerSizeWide": 3.2,
        "rowScaleWide": {"EFC (linear)": 1.8, "EFC (cubic)": 1.3},
        "dpi":        200,
    },
}
# endregion

## Imports

<details>
<summary>Pure analysis — reads stored data only, so no JAX / device setup is needed.</summary>

`schema_pop.final_states_array` recovers the EFC lanes, `reporting.scopeRejectionReport` gives the
divergence/scope audit on the population, and `utils.generateLatexTableInline` /
`utils.generate_latex_table_new` emit the two table shapes the manuscript needs (non-float for the
two-column Results body, float for the single-column Appendix).

</details>

In [ ]:
# region -> imports (analysis-only: reads stored .h5 data; no forward solve, so no JAX)
# ---- repo-root bootstrap: run from any cwd (make `library` importable + resolve the
# ---- relative notebookData/ + config/ paths). Walks up to the dir containing library/. ----
import os, sys
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "library")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import json
import corner                                    # posterior-cloud overlay

import library.utils as utils
from library.hdf5 import schema_pop              # final_states_array + BAD_RUN_SENTINEL (EFC recovery)
import library.postproc.reporting as reporting   # scopeRejectionReport (EFC population scope report)

print("comparison notebook: analysis-only (no forward solve)")
# endregion

## Setup — the shared task

<details>
<summary>Targets, parameter names and the prior box are read straight from the baseline `.h5`
files, and the two baselines are asserted to share them</summary>

so the comparison is guaranteed aligned to what each backend actually fit — no re-derivation. The
cell also prints each file's stored `runConfig` model/scenario, which is how the pre-rename
provenance (`sepsis.json` -> `sepsis_cubic.json`) stays visible, and computes the simulated-second
budget of one EFC calibration from the scenario stage list.

`sigmaRel` differs between the two baseline runs (0.005 for emcee, 0.02 for GD), so `J` and
`log_prob` are **not** comparable across them; only relative error, forward-solve count and wall
time are. Nothing downstream compares `J` across methods.

</details>

In [ ]:
# region -> setup: shared targets / prior box / provenance, and the EFC simulated-time budget
ATM = runConfig["analysis"]["atm"]

def obsOffset(name):
    """Atmospheric offset baked into absolute-pressure signals (gauge = raw - offset)."""
    return utils.obsOffset(name, ATM)

# the shared task definition comes from the runs themselves (identical across the baseline files)
_ref = runConfig["methods"][0]["path"]
with h5py.File(_ref, "r") as f:
    tgt         = f["targets"][:]                              # (nTargeted,) gauge-unit targets
    obsTargeted = list(f["observation_names"].asstr()[:])      # the fitted observation set
    param_names = list(f["param_names"].asstr()[:])            # the 16 calibration params
    bounds      = f["bounds"][:]                               # (P, 2) prior box
lo, hi = bounds[:, 0], bounds[:, 1]
span   = hi - lo
P      = len(param_names)

# provenance: every baseline file must name the same task (pre-rename names are expected here)
print(f"shared task: {P} parameters -> {len(obsTargeted)} targeted observations")
for m in runConfig["methods"]:
    with h5py.File(m["path"], "r") as f:
        rc = json.loads(f.attrs["runConfig"])
        assert list(f["param_names"].asstr()[:]) == param_names, f"{m['name']}: param_names mismatch"
        assert np.allclose(f["targets"][:], tgt),                f"{m['name']}: targets mismatch"
        assert f["bounds"][:].shape == bounds.shape,             f"{m['name']}: bounds shape mismatch"
        print(f"  {m['name']:6s} <- {os.path.basename(m['path'])} | stored provenance "
              f"{rc.get('model')} / {rc.get('scenario')} | sigmaRel {rc['inference']['sigmaRel']}")
print(f"  EFC    <- {runConfig['scenario']} / {runConfig['model']} "
      f"(pre-rename: sepsis.json / cvModel.json, identical content -- commit d2f3d0e)")

# one EFC calibration = one staged simulation; its cost in SIMULATED seconds comes from the
# scenario stage budget, not from a hard-coded number
def efcSimSeconds(scenarioName):
    sc  = utils.loadScenario(scenarioName)
    rt  = sc["shared"]["integration"]["runTime"]
    return sum(s.get("runsToIgnore", 0) + s.get("runsToSave", 0)
               for s in sc["calibration"]["stages"]) * rt


# --- observation <-> parameter pairing (EFC's one-to-one assignment) --------------------------
# The model's calibration block is the single definition of which parameter drives which
# observable, so the figure can align column i of the per-target panel with column i of the
# per-parameter panel instead of relying on the two file orders happening to agree. Anything
# without a partner (an observable no controller drives, a parameter whose target is absent from
# the fitted set) is appended after the paired slots rather than silently dropped.
_modelJSON   = utils.loadJSONfile(utils.configPath("models", runConfig["model"]))
targetForPar = {k: v["params"]["varTarget"] for k, v in _modelJSON["calibration"].items()
                if "targetValue" in v["params"]}
_obsIdx  = {o: j for j, o in enumerate(obsTargeted)}
_paired  = [(i, _obsIdx[targetForPar[p]]) for i, p in enumerate(param_names)
            if targetForPar.get(p) in _obsIdx]
parOrder = [i for i, _ in _paired] + [i for i in range(P) if i not in {i for i, _ in _paired}]
obsOrder = [j for _, j in _paired] + [j for j in range(len(obsTargeted))
                                      if j not in {j for _, j in _paired}]
nPaired  = len(_paired)
print(f"paired {nPaired}/{len(obsTargeted)} observations to their controlling parameter"
      + ("" if nPaired == len(obsTargeted) == P else
         f" | unpaired observations: {[obsTargeted[j] for j in obsOrder[nPaired:]]}"
         f" | unpaired parameters: {[param_names[i] for i in parOrder[nPaired:]]}"))

_pad = lambda seq, n: list(seq) + [None] * (n - len(seq))
_n   = max(len(obsOrder), len(parOrder))
pd.DataFrame({"observation": _pad([obsTargeted[j] for j in obsOrder], _n),
              "target":      _pad([round(float(tgt[j]), 4) for j in obsOrder], _n),
              "parameter":   _pad([param_names[i] for i in parOrder], _n),
              "paired":      [k < nPaired for k in range(_n)]})
# endregion

## Load the baseline runs

<details>
<summary>Each baseline `.h5` gives its flattened posterior / optima cloud, the per-iteration
observation trajectory, the MAP / mean equivalence observables, and the cost attrs</summary>

(`nEval`, `fitWall`, `settleRuns`, backend `diag`). The extra quantity built here is the
**independent-attempt distribution**: the final-iteration `max|rel err|` of every walker (emcee) or
every restart (GD). That is what makes the robustness column possible — an optimiser that finds a
good optimum in one restart out of sixty-four has not solved the problem in any usable sense,
because nothing in the output identifies which restart was the good one.

Simulated time is `nEval x settleRuns x runTime` — the hardware-independent unit of account,
directly comparable to the EFC stage budget computed above.

</details>

In [ ]:
# region -> load each baseline: cloud, obs trajectory, cost attrs, per-attempt error distribution
# An attempt whose final max|rel err| exceeds `convergedLimit` did not converge; it carries no more
# information than a NaN lane, so it is dropped here (like the sentinel-stamped lanes the population
# files already carry) rather than plotted as a 10^5 % error. The count is reported, never silent.
convLimit = runConfig["analysis"].get("convergedLimit")
runs = {}
for m in runConfig["methods"]:
    with h5py.File(m["path"], "r") as f:
        r = {
            "label":        m["label"],
            "attemptKind":  m["attempts"],
            "chain":        f["chain"][:],                      # (S, P) flattened cloud
            "obs_steps":    f["obs_steps"][:],                  # (nIter, nPop, nTargeted) gauge obs
            "theta_attempt": f["chain_steps"][-1],              # (nAttempt, P) endpoint of every
                                                                # walker / restart -- the parameter
                                                                # counterpart of obs_steps[-1]
            "equiv_theta":  f["equiv_theta"][:],                # (2, P) [MAP, mean]
            "equiv_obs":    f["equiv_obs"][:],                  # (2, nTargeted) obs at MAP / mean
            "equiv_labels": list(f["equiv_labels"].asstr()[:]),
            "nEval":        int(f.attrs["nEval"]),
            "fitWall":      float(f.attrs["fitWall"]),
            "settleRuns":   int(f.attrs["settleRuns"]),
            "runTime":      float(f.attrs["runTime"]),
            "accept":       float(f.attrs.get("accept", float("nan"))),
            "diag":         json.loads(f.attrs.get("diag", "{}")),
            "runConfig":    json.loads(f.attrs["runConfig"]),
        }
    # hardware-independent cost: total simulated seconds integrated to reach the answer
    r["simSeconds"] = r["nEval"] * r["settleRuns"] * r["runTime"]
    # independent-attempt distribution: final-iteration max|rel err| of every walker / restart
    r["attemptErr"] = np.nanmax(np.abs((r["obs_steps"][-1] - tgt) / tgt), axis=1) * 100.0
    r["attemptErrRaw"] = r["attemptErr"].copy()            # pre-filter, for the stored-diag check
    conv = (np.isfinite(r["attemptErr"]) if convLimit is None
            else np.isfinite(r["attemptErr"]) & (r["attemptErr"] <= convLimit))
    r["nNonConverged"] = int((~conv).sum())
    r["obs_steps"]     = r["obs_steps"][:, conv, :]
    r["theta_attempt"] = r["theta_attempt"][conv]
    r["attemptErr"]    = r["attemptErr"][conv]
    runs[m["name"]] = r

    a = r["attemptErr"]
    print(f"{m['name']:6s}: cloud {r['chain'].shape} | obs_steps {r['obs_steps'].shape} | "
          f"{r['nEval']:,} solves x {r['settleRuns']} settle x {r['runTime']:.0f}s "
          f"= {r['simSeconds']:.3g} sim-s | {r['fitWall']:.1f}s wall")
    print(f"        {len(a)} {r['attemptKind']}: max|rel err| best {a.min():.3f}% "
          f"median {np.median(a):.3f}% worst {a.max():.3f}%"
          + (f" | {r['nNonConverged']} non-converged (> {convLimit:g}%) dropped"
             if r["nNonConverged"] else ""))

# GD stores its own per-restart final error; cross-check it against the reconstruction above
_w = runs.get("gd", {}).get("diag", {}).get("worstRelFinal")
if _w is not None:
    assert np.allclose(np.sort(np.asarray(_w)), np.sort(runs["gd"]["attemptErrRaw"]), rtol=1e-6), \
        "gd: reconstructed per-restart error disagrees with the stored worstRelFinal"
    print("gd: per-restart error reconstruction matches the stored diag.worstRelFinal")
# endregion

## EFC lanes (consult-only)

<details>
<summary>EFC lanes recovered from stored population files — no EFC run happens here</summary>

One lane per entry in `runConfig["efc"]["populations"]`, each a **population**: N independent
random starts drawn across the *full* prior box, every one driven to the *same* fixed twin
targets. Verified in-cell, because this is what makes a lane the like-for-like counterpart of
GD's multi-start — same problem, different start. The control law is read from each file's own
stored config rather than its filename (several population files on disk are misnamed) and
cross-checked against the configured `label`; a mismatch prints a NOTE.

`amp_P_As` is an algebraic output rather than a stored state, so it is reconstructed as
`keep_max_P_As - keep_min_P_As`. That lets every EFC lane be scored on all 16 targets, exactly the
set the baselines were scored on — no observable is quietly dropped from one side of the comparison.

</details>

In [ ]:
# region -> EFC recovery: score every population lane on the same 16 targets from stored final states
def efcObsMatrix(path):
    """(obs matrix on `obsTargeted`, param matrix, finite mask) from a stored final_states table."""
    arr, stateNames = schema_pop.final_states_array(path)          # (M, S)
    sIdx = {n: i for i, n in enumerate(stateNames)}
    cols = []
    for o in obsTargeted:
        if o in sIdx:
            cols.append(arr[:, sIdx[o]] - obsOffset(o))
        elif o.startswith("amp_") and f"keep_max_{o[4:]}" in sIdx:
            # amplitude is algebraic (not a stored state): max - min, offsets cancel
            cols.append(arr[:, sIdx[f"keep_max_{o[4:]}"]] - arr[:, sIdx[f"keep_min_{o[4:]}"]])
        else:
            cols.append(np.full(arr.shape[0], np.nan))
    obsM   = np.stack(cols, axis=1)                                # (M, nTargeted)
    paramM = np.stack([arr[:, sIdx[p]] for p in param_names], axis=1)
    finite = (~np.any(arr <= schema_pop.BAD_RUN_SENTINEL + 1.0, axis=1)
              & ~np.any(~np.isfinite(arr), axis=1))
    return arr, sIdx, obsM, paramM, finite

def maxRel(obsM):
    return np.nanmax(np.abs((obsM - tgt) / tgt), axis=-1) * 100.0

efc = {}
for _pc in runConfig["efc"]["populations"]:
    with h5py.File(_pc["path"], "r") as f:
        popConf    = json.loads(f.attrs["conf"])
        popTiming  = json.loads(f.attrs["timing"])
        popTargets = f["final_states"]["target_avg_P_Vs"][:]

    # provenance: the control law comes from the scenario the run actually used, never the filename
    _scName = popConf["scenario"]
    _law    = _scName.replace("sepsis_", "").replace(".json", "")
    _label  = _pc.get("label", f"EFC ({_law}, population)")
    # The law comes from the stored config, never the filename or the display label. Any source
    # that disagrees is named once -- silently trusting the wrong one mislabels every downstream
    # table and figure.
    _disagree = ([f"label '{_pc['label']}'"] if "label" in _pc and _law not in _pc["label"] else []) \
              + ([f"filename '{os.path.basename(_pc['path'])}'"]
                 if _law not in os.path.basename(_pc["path"]) else [])
    print(f"EFC {_law} <- {os.path.basename(_pc['path'])} "
          f"({popConf['model']} / {_scName})")
    if _disagree:
        print(f"   NOTE: {' and '.join(_disagree)} disagree with the stored law '{_law}' "
              f"-- the stored config wins")

    _, _, obsPop, parPop, finitePop = efcObsMatrix(_pc["path"])

    # fairness precondition 1: every lane was driven to the SAME target, only the start differs
    _valid = popTargets > schema_pop.BAD_RUN_SENTINEL + 1.0
    assert np.unique(np.round(popTargets[_valid], 9)).size == 1, \
        f"{_law}: EFC lanes do not share a single target -- not a like-for-like multi-start"

    # fairness precondition 2: those targets are the ones the BASELINES fit. The population may come
    # from a different model/scenario file than the baselines, so this is checked, never assumed.
    _arrT, _namesT = schema_pop.final_states_array(_pc["path"])
    _iT = {n: i for i, n in enumerate(_namesT)}
    _mismatch, _nChecked = [], 0
    for _o, _t in zip(obsTargeted, tgt):
        _k = f"target_{_o}"
        if _k not in _iT:
            continue                                   # not persisted for this observable; skip
        _nChecked += 1
        _col = _arrT[:, _iT[_k]]
        _u = np.unique(np.round(_col[_col > schema_pop.BAD_RUN_SENTINEL + 1.0], 9))
        if not np.allclose(_u, _t):
            _mismatch.append((_o, float(_u[0]), float(_t)))
    assert not _mismatch, (f"{_law}: EFC targets differ from the baseline targets -- the task is NOT "
                           f"identical: {_mismatch}")

    # fairness precondition 3: the search box. Report any disagreement rather than hiding it.
    with h5py.File(_pc["path"], "r") as f:
        _popBounds = np.array(json.loads(f.attrs["problem"])["bounds"])
    _boxDiff = [(p, tuple(float(v) for v in _popBounds[i]), tuple(float(v) for v in bounds[i]))
                for i, p in enumerate(param_names) if not np.allclose(_popBounds[i], bounds[i])]
    print(f"   same task: {_nChecked}/{_nChecked} targets match the baselines, search box matches "
          f"on {len(param_names) - len(_boxDiff)}/{len(param_names)} parameters"
          + "".join(f"\n   box differs: {p} population {a} vs baseline {b}" for p, a, b in _boxDiff))

    # verbose=False: the full per-signal rejection tables are ~20 lines per lane and are almost
    # all zeros for these populations. The counts are still reported below -- a dropped run is
    # never silent -- and reporting.scopeRejectionReport(..., verbose=True) prints the detail.
    popScope = reporting.scopeRejectionReport(obsPop, parPop, obsTargeted, param_names,
                                              lim=runConfig["analysis"]["divergenceLimit"],
                                              verbose=False)
    relPop = np.full(obsPop.shape[0], np.inf)
    relPop[finitePop] = maxRel(obsPop[finitePop])
    # non-convergence == the sentinel case, one tier up: the lane survived the solve but ended so
    # far from target that its value is noise. Dropped from every downstream panel and table.
    nonConv   = finitePop & ~(relPop <= convLimit) if convLimit is not None else np.zeros_like(finitePop)
    converged = finitePop & ~nonConv
    efc[_law] = {
        "label":      _label,
        "short":      f"EFC ({_law})",
        "color":      _pc.get("color", "C2"),
        "law":        _law,
        "scenario":   _scName,
        "path":       _pc["path"],   # source file for the stored calibration trajectory
        "convMask":   converged,     # (nTotal,) lane mask; selects the same lanes as `obs`
        "attemptErr": relPop[converged],
        "nAttempts":  int(converged.sum()),
        "nNonConverged": int(nonConv.sum()),
        "nTotal":     int(obsPop.shape[0]),
        "wall":       popTiming["total_wall"] / popTiming["nrModels"],   # amortised per calibration
        "wallTotal":  popTiming["total_wall"],       # undivided: what the whole batch actually cost
        "simSeconds": efcSimSeconds(_scName),        # per calibration (one staged run)
        "cloud":      parPop[converged & (relPop < runConfig["efc"]["cloudThresh"])],
        "obs":        obsPop[converged],   # (nAttempt, nTargeted): every valid lane, target by target
        "params":     parPop[converged],   # (nAttempt, P): the same lanes, parameter by parameter
        "obsBest":    obsPop[np.argmin(np.where(converged, relPop, np.inf))],
    }
    e, _c = efc[_law], popScope.counts
    # every dropped lane is accounted for by reason; the error/cost summary itself lives in the
    # cost table and the attempt-spread table below, so it is not repeated here.
    print(f"   {e['nAttempts']}/{e['nTotal']} lanes kept"
          f" ({_c['dropped']} out of scope"
          + (f", {_c['sentinel']} sentinel-stamped" if _c["sentinel"] else "")
          + (f", {_c['nan']} NaN/Inf" if _c["nan"] else "")
          + (f"; {e['nNonConverged']} non-converged > {convLimit:g}%" if e["nNonConverged"] else "")
          + ")")
# endregion

## Cost, accuracy and attempt spread — the head-to-head table


<details>

Every entry is counted in ONE unit — a single 10 s forward solve, timed at `analysis.solveWall` s
by solver_compare (explicit Euler, dt 5e-4, one lane, one CPU). The table is therefore a solve
*count*, and the serial wall row is that count times `solveWall`.

**Solves / step** is what one iteration costs for one *attempt* — a walker, a restart, a lane. EFC
has no outer loop: its step is a single solve, so the row reads 1 and its whole cost sits in
**Steps**, the scenario stage budget expressed in 10 s runs. GD pays `1 + 2P`, the objective plus
the central-difference stencil over all sixteen parameters.

**Solves / calibration** is the like-for-like figure, what one independent calibration costs. A
sampler does not deliver one per walker — the ensemble is a single coupled chain — so its entry is
the whole run, written as the product that makes that explicit. **Total solves** covers the run as
configured, including each method's one-off overheads: emcee's warmup scoring, its initial ensemble
state and its burn-in rescues; GD's start screen, Laplace stencil and cloud rescore.

Settle runs are counted as one solve by default (`analysis.countSettleRuns`). The archived emcee
file stores `settleRuns` 3 against 1 for GD and for the EFC populations, which is a configuration
asymmetry rather than a property of the method.

**Best** is the lowest max|rel err| over the point set the method *delivers*: the final lane states
for a multi-start method, the post-burn posterior for the sampler. **Median** and the within-X% row
describe the spread over *independent* calibrations, so they are reported only where
`runConfig["methods"][i]["independent"]` is true.

</details>

In [ ]:
# region -> cost: solves per step / per calibration / totals, plus accuracy (+ optional LaTeX emit)
# ONE unit for every column: a single 10 s forward solve. solver_compare times that solve at
# analysis.solveWall s (explicit Euler, dt 5e-4, one lane, one CPU), so a hardware-independent cost
# is a solve COUNT, and the serial wall row is that count x solveWall.
#
#   Solves / step  what ONE iteration costs for ONE attempt (walker / restart / lane). EFC has no
#                  outer loop -- its step IS one 10 s solve, so the row is 1 and the whole cost
#                  sits in `Steps`. GD pays 1 + 2P: the objective plus the central-difference
#                  stencil over all P parameters, i.e. 32 of its 33 solves buy a gradient.
#   Steps          iterations taken; for EFC, the scenario stage budget expressed in 10 s runs.
#   Solves / cal.  the like-for-like number: what ONE independent calibration costs. A sampler
#                  does not deliver one per walker -- the ensemble is a single coupled chain -- so
#                  its entry is the whole run, written as the product that makes that explicit.
#   Total solves   the run as configured: every lane / walker / restart, plus the one-off overheads
#                  each method pays (emcee's warmup scoring, its burn-in rescues and its initial
#                  ensemble state; GD's start screen, Laplace stencil and cloud rescore, which its
#                  stored nEval already carries).
#
# Settle runs are NOT counted by default: the archived emcee file stores settleRuns 3 (30 simulated
# seconds per proposal) against 1 for GD and for the EFC populations, which is a configuration
# asymmetry rather than a property of the method. analysis.countSettleRuns False scores every
# method on a single 10 s settle; True restores each file's own stored value (and triples emcee).
solveWall   = runConfig["analysis"]["solveWall"]
levels      = runConfig["analysis"]["successLevels"]
countSettle = runConfig["analysis"].get("countSettleRuns", False)

def _within(a, t):  return 100.0 * float(np.mean(np.asarray(a) < t))
def _f(v):          return f"{v:,.0f}"

cost, order, perCalN, indepOf = {}, [], {}, {}

for law, e in efc.items():
    # steps = the stage budget in 10 s runs; simSeconds is that budget x the scenario runTime
    rt    = utils.loadScenario(e["scenario"])["shared"]["integration"]["runTime"]
    steps = int(round(e["simSeconds"] / rt))
    total = steps * e["nTotal"]                     # every lane the batch integrated, dropped ones too
    cost[e["label"]] = {
        "attempts":   f"{e['nAttempts']}",
        "perStep":    "1",
        "steps":      _f(steps),
        "perCal":     _f(steps),
        "total":      _f(total),
        "wallSerial": _f(total * solveWall),
        "wallMeas":   _f(e["wallTotal"]),
        "best":       f"{float(e['attemptErr'].min()):.2f}",
        "median":     f"{float(np.median(e['attemptErr'])):.2f}",
        **{f"within{lvl}": f"{_within(e['attemptErr'], lvl):.0f}" for lvl in levels},
    }
    order.append(e["label"]); perCalN[e["label"]] = steps; indepOf[e["label"]] = True

for m in runConfig["methods"]:
    r, indep = runs[m["name"]], m.get("independent", True)
    d, _inf  = r["diag"], r["runConfig"]["inference"]
    settle   = r["settleRuns"] if countSettle else 1
    nIt      = int(d["nSteps"])
    extra    = 0
    if m["name"] == "emcee":
        nW      = int(d["nWalkers"])
        perStep = settle
        perCal  = nW * nIt * settle
        perCalS  = _f(perCal)
        perStepS = _f(perStep)
        # one-off evaluations outside the sampler loop: the warmup cloud that defines the whitening
        # map, the initial ensemble state, and every burn-in walker rescue (all absent from nEval)
        _wc   = _inf.get("whiten", {})
        extra = ((_wc.get("warmup", 0) if (_wc.get("enabled") or _inf.get("init") == "warmup") else 0)
                 + nW + int(d.get("nRescued", 0)))
    else:
        stencil  = 1 + (2 * P if d.get("gradSource", "fd") == "fd" else 0)
        perStep  = stencil * settle
        perCal   = nIt * perStep + settle                      # + the step-0 objective evaluation
        perStepS = _f(perStep)
        perCalS  = _f(perCal)
    total = (r["nEval"] + extra) * settle
    if indep:
        bestErr = float(r["attemptErr"].min())
    else:
        # a sampler's delivered answer is the posterior, not its last iterate: score every
        # post-burn sample, so `Best` means the same thing here as it does for the other columns
        _b      = int(d.get("burnIn", 0))
        _post   = np.nanmax(np.abs((r["obs_steps"][_b:] - tgt) / tgt), axis=2) * 100.0
        bestErr = float(np.nanmin(_post))
    cost[r["label"]] = {
        # bare count: the caption names the unit (starts / walkers / restarts) once,
        # and `attemptKind` still labels the per-attempt rows everywhere else
        "attempts":   f"{len(r['attemptErr'])}",
        "perStep":    perStepS,
        "steps":      _f(nIt),
        "perCal":     perCalS,
        "total":      _f(total),
        "wallSerial": _f(total * solveWall),
        "wallMeas":   _f(r["fitWall"]),
        "best":       f"{bestErr:.2f}",
        # Spread over the method's own attempts. For GD and EFC these are independent
        # calibrations, so the rows read as a success rate; for MCMC the walkers are coupled by
        # construction (each proposal is built from the others' positions), so the same rows
        # describe the width of the posterior. The caption says which is which.
        "median":     f"{float(np.median(r['attemptErr'])):.2f}",
        **{f"within{lvl}": f"{_within(r['attemptErr'], lvl):.0f}" for lvl in levels},
    }
    order.append(r["label"]); perCalN[r["label"]] = perCal; indepOf[r["label"]] = indep

# (display label, LaTeX label, key) -- methods are the COLUMNS: the manuscript body is two-column,
# and four method columns fit where four method rows of eight metrics each would not.
ROWS = [
    ("Sample size",                           "Sample size",                             "attempts"),
    ("Solves / step",                         "Solves / step",                           "perStep"),
    ("Steps",                                 "Steps",                                   "steps"),
    ("Solves/cal",                            "Solves/cal",                              "perCal"),
    ("Total solves",                          "Total solves",                            "total"),
    ("Best max|rel err| %",                   "Best $|\\varepsilon_r|_{\\max}$ \\%",     "best"),
    ("Median max|rel err| %",                 "Median $|\\varepsilon_r|_{\\max}$ \\%",   "median"),
] + [(f"Samples < {lvl:g} %", f"Samples $<{lvl:g}$\\,\\%", f"within{lvl}") for lvl in levels]

costTable = pd.DataFrame({lab: [cost[lab][k] for _, _, k in ROWS] for lab in order},
                         index=[disp for disp, _, _ in ROWS])[order]
display(costTable)

# the claim that survives: what ONE independent calibration costs, in 10 s solves
_lo = min(perCalN.values())
print(f"\nsolves per independent calibration (cheapest = {_lo:,}):")
for lab, v in sorted(perCalN.items(), key=lambda kv: kv[1]):
    print(f"  {lab:34s} {v:12,}   {v / _lo:8.1f}x"
          + ("" if indepOf[lab] else "   (the whole run -- the walkers are one coupled chain)"))

if runConfig["paper"]["emit"].get("table", False):
    _tex  = lambda s: s.replace(",", "\\,").replace("%", "\\%")
    # the method labels are the widest cells in their own columns -- eleven rows of short numbers
    # sit under "EFC (linear)" -- so a parenthesised qualifier is stacked onto a second line
    _hdr  = lambda s: (f"\\shortstack[r]{{{s.split(' (')[0]}\\\\({s.split(' (')[1]}}}"
                       if " (" in s else s)
    body  = {tex: [_tex(cost[lab][k]) for lab in order] for _, tex, k in ROWS}
    latex = utils.generateLatexTableInline(
        body,
        [""] + [_hdr(lab) for lab in order],
        ref="tab:techniqueComparison",
        colSpec="l" + " r" * len(order),
        fontSize=runConfig["paper"].get("tableFontSize", "scriptsize"),
        caption=(
            "Cost and accuracy of the four calibrations on the identical task of "
            "Section~\\ref{sec:SimulationSetup}, every entry counted in one unit, a single "
            "ten-second forward solve timed in Table~\\ref{table:integratorComparison}. "
            "\\emph{Sample size} is each method's starts, walkers or restarts; the rows below "
            "\\emph{Best} describe the spread over them."))
    os.makedirs(runConfig["paper"]["dir"], exist_ok=True)
    _out = os.path.join(runConfig["paper"]["dir"], runConfig["paper"]["table"])
    with open(_out, "w") as fh:
        fh.write(latex)
    print(f"wrote {_out}")
# endregion

## Fit accuracy and parameter agreement — the comparison figure

<details>
<summary>One figure for the endpoint comparison: accuracy target by target, and where each
attempt's parameters landed</summary>

Both panels score the *same* populations — the EFC lanes at the end of their staged calibration,
the emcee walkers and the GD restarts at their final iteration — and nothing is filtered out, so a
method that succeeds only in a minority of its starts shows it here rather than hiding behind its
best answer. One colour per method, one figure legend, both shared with the convergence figure
below.

The panels are **transposed**: the categorical axis runs down the page and the measured quantity
across it. The sixteen target and parameter names then read horizontally, which is what makes the
figure legible at the manuscript's single-column width. The figure is drawn at that width
(`paper.figSize`), so its annotations are already at their final point size — a wide figure
downscaled into an 8.5 cm column renders 7 pt text at 2 pt.

**(a) Fit accuracy, target by target.** Grouped box plots over all 16 targets. Box position is
accuracy, box length is reliability: a long box means the fit depended on where the attempt started.

**(b) Parameter agreement.** The same attempts scored on the calibrated parameters, every attempt
drawn as a point, as its *fold* difference from the MCMC posterior median — `max(r, 1/r)`, so a
parameter recovered at twice the reference and one recovered at half it both plot at 2, and exact
agreement sits on the floor at 1. Folding halves the axis range and doubles the resolution near
agreement, at the cost of the direction of the miss. Drawn as a scatter rather than a summary
interval so the *shape* of each method's cloud is visible — which is what shows multi-modality: GD's
restarts fall into discrete clusters on several parameters rather than scattering smoothly.

Parameter endpoints come from `chain_steps[-1]`, the position of every walker and restart at its
final iteration — deliberately *not* the stored `chain`, which for GD is a Laplace cloud around the
optima rather than the optima themselves.

Rows (a) and (b) share an index: row *i* of each is the same EFC assignment — a parameter and the
observable its controller drives. Unpaired slots sit last.

</details>

In [ ]:
# region -> comparison figure: (a) fit accuracy, (b) parameter agreement
# The endpoint half of the comparison, drawn at the manuscript's single-column width
# (paper.figSize) so its annotations are already at their final point size -- a wide figure
# downscaled into an 8.5 cm column renders 7 pt text at 2 pt. Both panels are TRANSPOSED: the
# categorical axis runs down the page and the measured quantity across it, so the sixteen target
# and parameter names read horizontally instead of being rotated into a narrow column. The
# convergence trace is emitted as its own figure in the next cell.
#
# Both panels score the SAME independent attempts -- emcee walkers and GD restarts at their
# final iteration, EFC lanes at the end of their staged calibration -- and share one colour per
# method and one figure legend. They are ordered by `obsOrder` / `parOrder`, so row i of each
# is the same EFC assignment (a parameter and the observable its controller drives); unpaired slots
# sit last.
fs        = runConfig["analysis"].get("fontSize", 7)
foldClip  = runConfig["analysis"].get("foldClip", 1000.0)
axLim     = runConfig["analysis"].get("ylim", {})       # per-panel measure-axis limits; None = fit
band      = runConfig["analysis"]["errBand"]
labelsObs = [utils.labelsFor(obsTargeted, "latex")[j] for j in obsOrder]
labelsPar = [utils.labelsFor(param_names, "latex")[i] for i in parOrder]

lanes3 = [(e["label"], e["obs"], e["params"], e["color"]) for e in efc.values()]
for m in runConfig["methods"]:
    r = runs[m["name"]]
    lanes3.append((r["label"], r["obs_steps"][-1], r["theta_attempt"], m["color"]))

nS = len(lanes3); w = 0.8 / nS
legH, legL = [], []                                     # legend proxies, filled by panel (a)
yo = np.arange(len(obsOrder))
yp = np.arange(len(parOrder))
fig, (axA, axB) = plt.subplots(
    2, 1, figsize=tuple(runConfig["paper"]["figSize"]),
    gridspec_kw={"height_ratios": runConfig["paper"].get("heightRatios", [1.0, 0.95])})

# --- panel (a): fit error target by target, over every attempt ----------------------------------
for k, (name, obsM, _, colr) in enumerate(lanes3):
    errRel = (obsM - tgt) / tgt * 100.0                        # (nAttempt, nTargeted) rel error %
    cols   = [errRel[np.isfinite(errRel[:, j]), j] for j in obsOrder]
    bp = axA.boxplot(cols, positions=yo + (k - (nS - 1) / 2) * w, widths=w * 0.85,
                     orientation="horizontal", showfliers=False, patch_artist=True,
                     medianprops=dict(color="k", lw=0.5))
    for box in bp["boxes"]:
        box.set(facecolor=colr, alpha=0.6, edgecolor=colr, lw=0.45)
    for part in ("whiskers", "caps"):
        for ln in bp[part]:
            ln.set(color=colr, lw=0.45)
    legH.append(plt.Line2D([], [], ls="none", marker="s", color=colr, alpha=0.6, ms=4.5))
    legL.append(name)

axA.axvline(0.0, color="k", lw=0.6)
axA.axvline(band, color="r", ls="--", lw=0.6)
axA.axvline(-band, color="r", ls="--", lw=0.6)   # explained in the caption, not the legend
axA.set_yticks(yo)
axA.set_yticklabels(labelsObs, fontsize=fs)
axA.set_ylim(-0.6, len(obsOrder) - 0.4)
if runConfig["analysis"].get("errorScale", "linear") == "symlog":
    # linear inside the tolerance band, logarithmic outside it: the band doubles as the
    # linear/log crossover, so the sub-percent boxes stay readable beside a -12% outlier.
    axA.set_xscale("symlog", linthresh=band, linscale=1.0)
if axLim.get("fitError") is not None:
    axA.set_xlim(*axLim["fitError"])
axA.invert_yaxis()                                # first target at the top, reading order
axA.tick_params(axis="x", labelsize=fs)
axA.set_xlabel("relative error (%)", fontsize=fs)
axA.grid(axis="y", ls=":", lw=0.4, color="0.9")
axA.set_title("(a)  Fit accuracy per target", fontsize=fs + 1, fontweight="bold")

# --- panel (b): how far every individual attempt landed from the MCMC posterior median ----------
# The deviation is FOLDED: a parameter recovered at twice the reference and one recovered at half
# it are both "out by a factor of 2", so the ratio is replaced by max(r, 1/r). That halves the axis
# range and puts exact agreement on the floor at 1, at the cost of the direction of the miss.
# Points beyond `foldClip` are off-scale artifacts: they are hidden HERE ONLY and counted in the
# panel -- every attempt still appears in (a) and in every table.
jit   = np.random.default_rng(0)
refMC = np.median(runs["emcee"]["chain"], axis=0)               # (P,) the MCMC reference parameter
folds = []
for _, _, parM, _ in lanes3:
    r = parM / refMC                                            # (nAttempt, P)
    folds.append(np.where(r > 0, np.maximum(r, 1.0 / np.where(r == 0, np.nan, r)), np.nan))

# The cap: an explicit ylim.fold wins, otherwise fit the data but never past foldClip.
_below  = np.concatenate([f[np.isfinite(f) & (f <= foldClip)].ravel() for f in folds])
foldMax = float(_below.max()) if _below.size else 0.0
foldTop = (axLim["fold"] if axLim.get("fold") is not None
           else (min(foldClip, foldMax * 1.15) if foldMax > 0 else foldClip))

saturated = {}
for k, ((name, _, _, colr), fold) in enumerate(zip(lanes3, folds)):
    base = yp + (k - (nS - 1) / 2) * w
    nSat = 0
    for j, pi in enumerate(parOrder):
        v   = fold[:, pi]
        ok  = np.isfinite(v)
        sat = ok & (v > foldTop)                                # off the end of the axis
        nSat += int(sat.sum())
        inr = ok & ~sat
        axB.plot(v[inr], base[j] + jit.normal(0, w * 0.22, int(inr.sum())), ".",
                 color=colr, ms=1.8, alpha=0.5, mec="none")
        # saturated points are PARKED ON the cap as open carets, so a capped panel still shows
        # that attempts exist beyond it (and how many, and for which parameter). They stay inside
        # the axes -- an artist drawn outside makes tight_layout shrink every panel to fit it.
        axB.plot(np.full(int(sat.sum()), foldTop),
                 base[j] + jit.normal(0, w * 0.22, int(sat.sum())), ">", color=colr, ms=3.0,
                 alpha=0.9, mfc="none", mew=0.7)
    if nSat:
        saturated[name] = nSat

axB.axvline(1.0, color="k", lw=0.6, zorder=0)
axB.set_xscale("log")
# set_xlim goes AFTER set_xticks -- a tick list running past the view makes matplotlib widen a
# log axis back out to cover it.
axB.set_xticks([t for t in (1.0, 1.1, 1.5, 2.0, 5.0, 10.0, 50.0, 100.0, 500.0, foldClip)
                if t <= foldTop])
axB.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
axB.xaxis.set_minor_formatter(plt.NullFormatter())
axB.set_xlim(0.97, foldTop)             # the cap IS the right edge -- carets sit on it
axB.grid(axis="x", ls=":", lw=0.4, color="0.8")
axB.grid(axis="y", ls=":", lw=0.4, color="0.9")
axB.set_yticks(yp)
axB.set_yticklabels(labelsPar, fontsize=fs)
axB.set_ylim(-0.6, len(parOrder) - 0.4)
axB.invert_yaxis()
axB.tick_params(axis="x", labelsize=fs)
axB.set_xlabel("fold difference from the MCMC median", fontsize=fs)
# short titles: a title wider than its own axes runs into the neighbouring panel
axB.set_title("(b)  Parameter agreement", fontsize=fs + 1, fontweight="bold")
if saturated:
    print(f"panel (b): attempts parked on the {foldTop:.3g}x cap as open carets on the right edge "
          f"(still in (a) and every table, at their true value): {saturated}")

# The legend lives outside every panel. tight_layout cannot see a figure legend, so the strip it
# needs is reserved explicitly, sized from the number of rows it will occupy.
_ncol = 2
_band = (int(np.ceil(len(legH) / _ncol)) * (fs + 4) + 8) / (fig.get_size_inches()[1] * 72)
fig.tight_layout(rect=(0, _band, 1, 1.0), h_pad=1.4)
fig.legend(legH, legL, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=_ncol,
           fontsize=fs, framealpha=0.9)

_pf = runConfig["paper"]
if _pf["emit"].get("figure"):
    os.makedirs(_pf["imageDir"], exist_ok=True)
    _out = os.path.join(_pf["imageDir"], _pf["figure"])
    fig.savefig(_out, dpi=_pf["dpi"], bbox_inches="tight")
    print(f"figure -> {_out}")
plt.show()
# endregion

## Fit accuracy and parameter agreement — the full-width variant

<details>
<summary>The same comparison figure drawn at the manuscript's full text width, for when the tightest population is the one that disappears.</summary>

A horizontal box's length runs along the data axis, so the page a spread covers scales with the
figure width. EFC (linear) has the smallest spread of the four, and at the 8.5 cm column width its
box collapses to a hairline beside MCMC's and GD's. Drawn at full text width the same box is twice
as long and reads as a narrow box.

Nothing about the data, the panels or the colours changes. The knobs are `paper.figSizeWide`,
`paper.markScaleWide` (line weights and point sizes), `paper.boxFracWide` (box thickness along the
categorical axis) and `paper.fontSizeWide`. The PNG goes to `paper.figureWide`, a different file
from the column-width cell above, so the two can be run in any order.

</details>


In [ ]:
# region -> comparison figure, WIDE: same two panels at full text width
# The alternative to the column-width cell above, for when the tightest population is the one that
# disappears. Same data, same panels, same colours; the canvas, the weight of the marks and the
# row allocation differ.
#
# Two separate problems are solved here, and width only solves the first. A horizontal box's LENGTH
# lies along the data axis, so doubling the figure width doubles the page a given spread covers
# (`paper.figSizeWide`). But the EFC (linear) population's per-target IQR is ~0.009%, against ~0.65%
# for MCMC -- a factor of 71 -- so its box renders a couple of pixels wide however wide the canvas
# is, and no axis stretch recovers it. Its POSITION is therefore marked explicitly: every series
# gets a filled marker at its median (`paper.medianMarkerWide`), so a box too narrow to see still
# reads as a definite locus, and the tight populations are given thicker rows
# (`paper.rowScaleWide`) so the eye lands on them. Slot heights are reallocated in proportion to
# those scales, so the series still tile one row without overlapping.
#
# Written to `paper.figureWide`, a different file from the column-width cell, so both cells can be
# run in any order without either clobbering the other's PNG.
fsW      = runConfig["paper"].get("fontSizeWide", runConfig["analysis"].get("fontSize", 7))
mScale   = runConfig["paper"].get("markScaleWide", 2.0)
boxFrac  = runConfig["paper"].get("boxFracWide", 0.95)
medMark  = runConfig["paper"].get("medianMarkerWide", "D")
medMs    = runConfig["paper"].get("medianMarkerSizeWide", 3.2)
rowScale = runConfig["paper"].get("rowScaleWide", {})
foldClip = runConfig["analysis"].get("foldClip", 1000.0)
axLim    = runConfig["analysis"].get("ylim", {})
band     = runConfig["analysis"]["errBand"]

lanesW = [(e["label"], e["obs"], e["params"], e["color"]) for e in efc.values()]
for m in runConfig["methods"]:
    r = runs[m["name"]]
    lanesW.append((r["label"], r["obs_steps"][-1], r["theta_attempt"], m["color"]))

# Slot heights in proportion to each series' row scale, tiling the same 0.8 of a row the equal
# split used: with every scale at 1.0 this reproduces the (k - (n-1)/2) * w layout exactly.
scalesW = np.array([rowScale.get(name, 1.0) for name, _, _, _ in lanesW], dtype=float)
hW      = 0.8 * scalesW / scalesW.sum()                 # slot height per series
ctrW    = -0.4 + np.cumsum(hW) - hW / 2                 # slot centre offset per series
nSW     = len(lanesW)
legHW, legLW = [], []
yoW = np.arange(len(obsOrder))
ypW = np.arange(len(parOrder))
figW, (axAW, axBW) = plt.subplots(
    2, 1, figsize=tuple(runConfig["paper"].get("figSizeWide", [6.9, 6.6])),
    gridspec_kw={"height_ratios": runConfig["paper"].get("heightRatios", [1.0, 0.95])})

# --- panel (a): fit error target by target, over every attempt ----------------------------------
for k, (name, obsM, _, colr) in enumerate(lanesW):
    errRel = (obsM - tgt) / tgt * 100.0
    cols   = [errRel[np.isfinite(errRel[:, j]), j] for j in obsOrder]
    pos    = yoW + ctrW[k]
    bp = axAW.boxplot(cols, positions=pos, widths=hW[k] * boxFrac,
                      orientation="horizontal", showfliers=False, patch_artist=True,
                      medianprops=dict(color="k", lw=0.5 * mScale))
    for box in bp["boxes"]:
        box.set(facecolor=colr, alpha=0.6, edgecolor=colr, lw=0.45 * mScale)
    for part in ("whiskers", "caps"):
        for ln in bp[part]:
            ln.set(color=colr, lw=0.45 * mScale)
    # the median, marked explicitly: a box narrower than the axis can resolve still shows WHERE
    # the population sits. Dark edge so the marker reads against its own translucent box.
    meds = [np.median(c) if c.size else np.nan for c in cols]
    axAW.plot(meds, pos, ls="none", marker=medMark, ms=medMs, color=colr,
              mec="k", mew=0.35 * mScale, zorder=5)
    legHW.append(plt.Line2D([], [], ls="none", marker=medMark, color=colr, mec="k", mew=0.35,
                            ms=4.5))
    legLW.append(name)

axAW.axvline(0.0, color="k", lw=0.6)
axAW.axvline(band, color="r", ls="--", lw=0.6)
axAW.axvline(-band, color="r", ls="--", lw=0.6)
axAW.set_yticks(yoW)
axAW.set_yticklabels([utils.labelsFor(obsTargeted, "latex")[j] for j in obsOrder], fontsize=fsW)
axAW.set_ylim(-0.6, len(obsOrder) - 0.4)
if runConfig["analysis"].get("errorScale", "linear") == "symlog":
    axAW.set_xscale("symlog", linthresh=band, linscale=1.0)
if axLim.get("fitError") is not None:
    axAW.set_xlim(*axLim["fitError"])
axAW.invert_yaxis()
axAW.tick_params(axis="x", labelsize=fsW)
axAW.set_xlabel("relative error (%)", fontsize=fsW)
axAW.grid(axis="y", ls=":", lw=0.4, color="0.9")
axAW.set_title("(a)  Fit accuracy per target", fontsize=fsW + 1, fontweight="bold")

# --- panel (b): how far every individual attempt landed from the MCMC posterior median ----------
jitW  = np.random.default_rng(0)
refMC = np.median(runs["emcee"]["chain"], axis=0)
foldsW = []
for _, _, parM, _ in lanesW:
    r = parM / refMC
    foldsW.append(np.where(r > 0, np.maximum(r, 1.0 / np.where(r == 0, np.nan, r)), np.nan))

_below  = np.concatenate([f[np.isfinite(f) & (f <= foldClip)].ravel() for f in foldsW])
foldMax = float(_below.max()) if _below.size else 0.0
foldTop = (axLim["fold"] if axLim.get("fold") is not None
           else (min(foldClip, foldMax * 1.15) if foldMax > 0 else foldClip))

saturatedW = {}
for k, ((name, _, _, colr), fold) in enumerate(zip(lanesW, foldsW)):
    base = ypW + ctrW[k]
    nSat = 0
    for j, pi in enumerate(parOrder):
        v   = fold[:, pi]
        ok  = np.isfinite(v)
        sat = ok & (v > foldTop)
        nSat += int(sat.sum())
        inr = ok & ~sat
        axBW.plot(v[inr], base[j] + jitW.normal(0, hW[k] * 0.22, int(inr.sum())), ".",
                  color=colr, ms=1.8 * mScale, alpha=0.5, mec="none")
        axBW.plot(np.full(int(sat.sum()), foldTop),
                  base[j] + jitW.normal(0, hW[k] * 0.22, int(sat.sum())), ">", color=colr,
                  ms=3.0 * mScale, alpha=0.9, mfc="none", mew=0.7 * mScale)
    if nSat:
        saturatedW[name] = nSat

axBW.axvline(1.0, color="k", lw=0.6, zorder=0)
axBW.set_xscale("log")
axBW.set_xticks([t for t in (1.0, 1.1, 1.5, 2.0, 5.0, 10.0, 50.0, 100.0, 500.0, foldClip)
                 if t <= foldTop])
axBW.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:g}"))
axBW.xaxis.set_minor_formatter(plt.NullFormatter())
axBW.set_xlim(0.97, foldTop)
axBW.grid(axis="x", ls=":", lw=0.4, color="0.8")
axBW.grid(axis="y", ls=":", lw=0.4, color="0.9")
axBW.set_yticks(ypW)
axBW.set_yticklabels([utils.labelsFor(param_names, "latex")[i] for i in parOrder], fontsize=fsW)
axBW.set_ylim(-0.6, len(parOrder) - 0.4)
axBW.invert_yaxis()
axBW.tick_params(axis="x", labelsize=fsW)
axBW.set_xlabel("fold difference from the MCMC median", fontsize=fsW)
axBW.set_title("(b)  Parameter agreement", fontsize=fsW + 1, fontweight="bold")
if saturatedW:
    print(f"panel (b): attempts parked on the {foldTop:.3g}x cap as open carets on the right edge "
          f"(still in (a) and every table, at their true value): {saturatedW}")

# per-target IQR width, the number that decides whether a box can be seen at all
for (name, obsM, _, _) in lanesW:
    e = (obsM - tgt) / tgt * 100.0
    q = np.nanpercentile(e, [25, 75], axis=0)
    print(f"{name:13s} per-target IQR width (%): median {np.median(q[1]-q[0]):.4f}  "
          f"max {np.nanmax(q[1]-q[0]):.4f}")

# The legend strip is reserved explicitly, as in the column-width cell; at full width the four
# entries fit on one row.
_ncolW = 4
_bandW = (int(np.ceil(len(legHW) / _ncolW)) * (fsW + 4) + 8) / (figW.get_size_inches()[1] * 72)
figW.tight_layout(rect=(0, _bandW, 1, 1.0), h_pad=1.4)
figW.legend(legHW, legLW, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=_ncolW,
            fontsize=fsW, framealpha=0.9)

_pf = runConfig["paper"]
if _pf["emit"].get("figure"):
    os.makedirs(_pf["imageDir"], exist_ok=True)
    _outW = os.path.join(_pf["imageDir"], _pf.get("figureWide", "techniqueComparisonWide.png"))
    figW.savefig(_outW, dpi=_pf["dpi"], bbox_inches="tight")
    print(f"figure -> {_outW}")
plt.show()
# endregion

## Convergence to target — the trace figure

<details>
<summary>How each population collapses onto the targets, on a common normalised axis — the
companion to the cost table</summary>

Its own figure, placed under the cost table in the manuscript: the trace is read against the cost
accounting, and stacking it under the two endpoint panels made a figure taller than the text block.
Same colours, same order and the same converged attempts as the comparison figure, drawn at the
same single-column width (`paper.figSizeConvergence`).

Each curve is the ensemble median of the per-attempt `mean|rel err|`, with a shaded band over the
independent attempts (`analysis.convBand`). The baselines iterate, one entry per walker or restart;
EFC does not — one staged calibration *is* one simulation — so its curve is read from the stored
trajectory (`runConfig["efc"]["trajDataset"]`), scored on the same 16 targets over the same
converged lanes. The median rather than the mean, because a handful of EFC lanes excurse to
~1e17 % mid-run before recovering, and an ensemble mean would be theirs alone; on the baselines the
two agree to within 14 %. Progress is normalised to [0,1] because the backends run different
iteration counts and the two populations different stage budgets, so the comparison is of *shape*
— not of per-iteration cost, which the cost table above reports honestly. The printed endpoint
check re-derives each EFC curve's final value from the final-state table: the two must agree, or
the trajectory and the scoring have drifted apart.

</details>

In [ ]:
# region -> convergence figure: ensemble fit error against each method's own normalised progress
# The convergence half of the comparison, emitted as its OWN figure (paper.figureConvergence) and
# placed under the cost table in the manuscript, where it is read against the cost accounting.
# Drawn at the same single-column width as the comparison figure, so its annotations are already at
# their final point size, and it reuses that figure's colours and lane order.
#
# The baselines iterate (`obs_steps` is (nIter, nPop, nTargeted), one entry per walker / restart);
# EFC does not -- one staged calibration IS one simulation -- so its curve is read from the stored
# trajectory, scored on the same 16 targets over the same converged lanes the other panels use.
# Progress is normalised because the backends run different iteration counts and the two populations
# different stage budgets: the comparison is of SHAPE, not of per-iteration cost (that is the table).
fs         = runConfig["analysis"].get("fontSize", 7)
band       = runConfig["analysis"]["errBand"]
qLoC, qHiC = runConfig["analysis"].get("convBand", [25, 75])
trajDset   = runConfig["efc"].get("trajDataset", "raw_coarse")

def efcTrajErr(path, mask, dset=trajDset):
    """Per-lane mean|rel err| (%) trajectory, (nLane, nSample), on the same 16 targets as everything
    else here. Mirrors `efcObsMatrix` column by column (gauge offset, algebraic amplitude), so the
    last sample of the curve reproduces the lane's final-state score by construction."""
    with h5py.File(path, "r") as h:
        sIdx = {n: i for i, n in enumerate(h["raw_signal_names"].asstr()[:])}
        cols = []
        for o in obsTargeted:
            if o in sIdx:
                cols.append(h[dset][:, :, sIdx[o]][mask] - obsOffset(o))
            elif o.startswith("amp_") and f"keep_max_{o[4:]}" in sIdx:
                # amplitude is algebraic (not a stored signal): max - min, offsets cancel
                cols.append(h[dset][:, :, sIdx[f"keep_max_{o[4:]}"]][mask]
                            - h[dset][:, :, sIdx[f"keep_min_{o[4:]}"]][mask])
            else:
                cols.append(np.full((int(mask.sum()), h[dset].shape[1]), np.nan))
    obsT = np.stack(cols, axis=-1)                                  # (nLane, nSample, nTargeted)
    return np.nanmean(np.abs(obsT - tgt) / np.abs(tgt), axis=2) * 100.0

# same order and same colour as the comparison figure, so the two read together
traces = []
for e in efc.values():
    trE = efcTrajErr(e["path"], e["convMask"])
    fsE = np.nanmedian(np.nanmean(np.abs(e["obs"] - tgt) / np.abs(tgt), axis=1)) * 100.0
    print(f"{e['short']:13s} {trE.shape[0]} lanes x {trE.shape[1]} samples | endpoint "
          f"{np.nanmedian(trE[:, -1]):.3f}% vs final-state median {fsE:.3f}% (must agree)")
    traces.append((e["label"], e["color"], trE))
for m in runConfig["methods"]:
    r = runs[m["name"]]
    traces.append((r["label"], m["color"],
                   (np.nanmean(np.abs(r["obs_steps"] - tgt) / np.abs(tgt), axis=2) * 100.0).T))

# NOT transposed, unlike the endpoint panels: a trace against progress reads left to right, and
# turning it on its side costs more legibility than the row-label alignment buys.
figC, axC = plt.subplots(figsize=tuple(runConfig["paper"].get("figSizeConvergence", [3.35, 2.6])))
legHC, legLC = [], []
for name, colr, tr in traces:
    prog        = np.linspace(0, 1, tr.shape[1])                    # normalised progress
    lo, mid, hi = np.nanpercentile(tr, [qLoC, 50, qHiC], axis=0)
    # median rather than mean: a handful of EFC lanes excurse to ~1e17 % mid-run before recovering,
    # and an ensemble mean would be theirs alone (on the baselines the two agree to within 14%).
    axC.fill_between(prog, lo, hi, color=colr, alpha=0.15, lw=0)
    axC.plot(prog, mid, "-", lw=0.9, color=colr)
    legHC.append(plt.Line2D([], [], ls="-", lw=1.2, color=colr))
    legLC.append(name)
axC.axhline(band, color="r", ls="--", lw=0.6)
axC.set_yscale("log")
axC.set_xlim(0.0, 1.0)
axC.tick_params(axis="both", labelsize=fs)
axC.set_xlabel("normalised progress (0 = start, 1 = end)", fontsize=fs)
axC.set_ylabel("ensemble-median\nmean|rel err| (%)", fontsize=fs)
axC.grid(ls=":", lw=0.4, color="0.9")
# the band is named in the caption, not the title -- a title wider than the panel overhangs the
# column edge at this figure width
axC.set_title("Convergence to target", fontsize=fs + 1, fontweight="bold")

# The legend lives outside the axes. tight_layout cannot see a figure legend, so the strip it
# needs is reserved explicitly, sized from the number of rows it will occupy.
_ncolC = 2
_bandC = (int(np.ceil(len(legHC) / _ncolC)) * (fs + 4) + 8) / (figC.get_size_inches()[1] * 72)
figC.tight_layout(rect=(0, _bandC, 1, 1.0))
figC.legend(legHC, legLC, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=_ncolC,
            fontsize=fs, framealpha=0.9)

_pf = runConfig["paper"]
if _pf["emit"].get("figureConvergence"):
    os.makedirs(_pf["imageDir"], exist_ok=True)
    _outC = os.path.join(_pf["imageDir"], _pf["figureConvergence"])
    figC.savefig(_outC, dpi=_pf["dpi"], bbox_inches="tight")
    print(f"figure -> {_outC}")
plt.show()
# endregion